In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Load CSV data
df = pd.read_csv('2010_Chronic_Conditions.csv')
df.head()

In [ ]:
# Total Medicare Cost is calculated as the sum of Part A, Part B and Part D
df['Total_Cost'] = (
    df['AVE_PA_PAY_PA_EQ_12'].fillna(0) +  # Part A 
    df['AVE_CA_PAY_PB_EQ_12'].fillna(0) +  # Part B 
    df['AVE_OP_PAY_PB_EQ_12'].fillna(0) +  # Part B 
    df['AVE_PDE_CST_PD_EQ_12'].fillna(0)   # Part D 
)

In [ ]:
# Chronic condition columns
condition_cols = [col for col in df.columns if col.startswith('CC_')]

# Demographic columns
demographic_cols = ['BENE_SEX_IDENT_CD', 'BENE_AGE_CAT_CD']

# Utilization metrics 
utilization_cols = [
    'AVE_IP_ADM_PA_EQ_12',     # Inpatient admits
    'AVE_SNF_DAYS_PA_EQ_12',   # Skilled nursing facility days
    'AVE_CA_VST_PB_EQ_12',     # Physician visits
    'AVE_OP_VST_PB_EQ_12',     # Outpatient visits
    'AVE_PDE_PD_EQ_12'         # Prescription fills
]

In [ ]:
# Combine all feature columns
feature_cols = demographic_cols + condition_cols + utilization_cols

In [ ]:
#Look at the distribution of the data
plt.figure(figsize=(10, 5))
sns.histplot(df['Total_Cost'], bins=100, kde=True)
plt.title("Distribution of Total Medicare Cost")
plt.xlabel("Total Annual Medicare Cost ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Filter out Medicare Costs equal to 0
df_filtered = df[df['Total_Cost'] > 0]

In [ ]:
# Create feature matrix
X = df_filtered[feature_cols].copy()

# Fill missing values in binary flags and utilization columns with 0
X = X.fillna(0)

In [ ]:
# Apply log(1 + x) transformation to stabilize variance and reduce skew
y = df_filtered['Total_Cost']
y_log = np.log1p(y)

In [ ]:
#Distribution of the data after log transformation
plt.figure(figsize=(10, 5))
sns.histplot(y_log, bins=100, kde=True)
plt.title("Distribution of Total Medicare Cost")
plt.xlabel("Total Annual Medicare Cost ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [ ]:
# Linear Regression 
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

In [ ]:
# Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Create a function to evaluate models
def evaluate_model(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))
    }

In [ ]:
# Evaluate both models
lr_eval = evaluate_model(y_test, y_pred_lr)
rf_eval = evaluate_model(y_test, y_pred_rf)

In [ ]:
# Convert log-scale predictions back to dollar cost
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_rf)

In [ ]:
#Visualize Model Performance
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test_actual, y=y_pred_actual, alpha=0.3)
plt.plot([y_test_actual.min(), y_test_actual.max()],
         [y_test_actual.min(), y_test_actual.max()],
         'r--', label='Perfect Prediction')
plt.xlabel("Actual Total Medicare Cost")
plt.ylabel("Predicted Total Medicare Cost")
plt.title("Actual vs Predicted Medicare Cost (Random Forest)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Get and sort feature importances from the Random Forest model
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importances_sorted = feature_importances.sort_values(ascending=False)

In [ ]:
# Plot top 15 features
plt.figure(figsize=(10, 6))
sns.barplot(
    x=feature_importances_sorted.head(15),
    y=feature_importances_sorted.head(15).index,
    palette='viridis'
)
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
#Output Evaluation Scores
lr_eval, rf_eval

In [ ]:
#Format Evaluation Outputs
lr_eval = {'R2': 0.87, 'MAE': 0.14, 'RMSE': 0.20}
rf_eval = {'R2': 0.93, 'MAE': 0.09, 'RMSE': 0.14}

# Combine into one DataFrame
eval_df = pd.DataFrame({
    'Linear Regression': lr_eval,
    'Random Forest': rf_eval
})

In [ ]:
eval_df